# Fault tolerance: killing the orchestrator

**This notebook sends `SIGKILL` to the running orchestrator container, then starts it again.** Run it only when nothing
else needs the service. Any synchronous request in flight at that moment fails, as it would in a real crash.

Asynchronous jobs live in a durable SQLite store, and a running job holds a lease that its owner renews every 10 s. If the
owner dies, the lease expires after 30 s and the job is requeued. Because each job carries a seed, re-running it gives
identical counts. This takes about a minute.

In [1]:
import subprocess
from pathlib import Path

from qputwin_demo import *

connect()

PROJECT = Path.cwd().parent

def compose(*args):
    return subprocess.run(["docker", "compose", *args], cwd=PROJECT, check=True, capture_output=True, text=True).stdout.strip()

container = compose("ps", "-q", "orchestrator")
print("Orchestrator container:", container[:12])

QPUTwin at http://localhost:8000: ready, 4 twins reachable
Orchestrator container: c98f18eb1226


## A reference result

First, run the long job normally, so there is something to compare with. It takes a few seconds.

In [2]:
long_job = scrambler(27, 10)
start = time.time()
reference = run("ionq_forte", long_job, seed=4242, shots=512)
duration = time.time() - start
print(f"Took {duration:.1f} s. Distinct outcomes: {len(reference['result']['counts'])}")

Took 8.1 s. Distinct outcomes: 512


## Submit, then crash

One long job takes the slot, and three more wait behind it.

In [3]:
running = submit("ionq_forte", long_job, seed=4242, shots=512)
waiting = [submit("ionq_forte", ghz(10), seed=i)["job_id"] for i in range(3)]

while job(running["job_id"])["status"] != "RUNNING":
    time.sleep(0.05)
print("Long job:", job(running["job_id"])["status"], "| Queue:", queues()["ionq_forte"])

Long job: RUNNING | Queue: {'queued': 3, 'running': 1, 'capacity': 1}


In [4]:
print(subprocess.run(["docker", "kill", "--signal=KILL", container], capture_output=True, text=True).stdout[:12], "killed")
try:
    connect()
except RuntimeError as error:
    print("As expected:", error)

c98f18eb1226 killed
As expected: QPUTwin is not reachable at http://localhost:8000. Start it from the project root with: docker compose up -d --wait


## Restart and recover

In [5]:
subprocess.run(["docker", "start", container], check=True, capture_output=True)
for _ in range(100):
    try:
        connect()
        break
    except (RuntimeError, httpx.HTTPError):
        time.sleep(0.3)

start = time.time()
trace = []
while True:
    status = job(running["job_id"])
    trace.append((round(time.time() - start, 1), status["status"], status.get("attempts")))
    if status["status"] in ("COMPLETED", "FAILED"):
        break
    time.sleep(1)

print(f"Recovered {time.time() - start:.1f} s after restart.")
pd.DataFrame(trace, columns=["seconds", "status", "attempts"]).drop_duplicates(["status", "attempts"])

QPUTwin at http://localhost:8000: ready, 4 twins reachable


Recovered 38.1 s after restart.


,seconds,status,attempts
0,0.0,RUNNING,1
30,30.1,RUNNING,2
38,38.1,COMPLETED,2


## Nothing was lost

The long job ran twice, and its counts match the reference exactly. The queued jobs were never at risk, because they were
already in the durable store.

With monitoring running, the **Jobs re-executed after a lost lease** panel on the Workload dashboard rises by one. Running this notebook several times within ten minutes also fires the `ContainerRestarting` alert, which is the alert doing its job.

In [6]:
recovered = job(running["job_id"])
print("Long job status:   ", recovered["status"])
print("Attempts:          ", recovered["attempts"])
print("Same counts as the reference:", recovered["result"]["counts"] == reference["result"]["counts"])

others = [job(i, wait=30) for i in waiting]
print("Queued jobs:       ", [j["status"] for j in others])

Long job status:    COMPLETED
Attempts:           2
Same counts as the reference: True
Queued jobs:        ['COMPLETED', 'COMPLETED', 'COMPLETED']
